In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%pip install brian2 brian2hears librosa numpy scipy matplotlib setuptools

import logging
logging.getLogger('brian2').setLevel(logging.ERROR)

import brian2
brian2.prefs.codegen.target = 'numpy'

import numpy as np
import os
import re
import glob
import random
import tensorflow as tf
from scipy import signal
from math import gcd
from tensorflow.keras.callbacks import (
    ModelCheckpoint, EarlyStopping, CSVLogger, ReduceLROnPlateau
)
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker


SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

BASE_DIR = "/content/drive/MyDrive/p&d/fase_2/"
CKPT_DIR = os.path.join(BASE_DIR, "checkpoints_v3/")
os.makedirs(CKPT_DIR, exist_ok=True)

PAD_TRAIN_EEG = os.path.join(BASE_DIR, "data/64 Channel Biosemi unprocessed data - train/*/*/*.npz")
PAD_TEST_EEG  = os.path.join(BASE_DIR, "data/test_data/test_data/*/*.npz")
PAD_AUDIO     = os.path.join(BASE_DIR, "data/preprocessed/audio/cnn/")

# Unieke bestandsnaam per checkpoint
PAD_CHECKPOINT_EPOCH = os.path.join(
    CKPT_DIR,
    "v3_epoch{epoch:02d}_valacc{val_accuracy:.4f}_valloss{val_loss:.4f}.keras"
)
PAD_BEST_MODEL  = os.path.join(BASE_DIR, "hybrid_v3_BEST.keras")
PAD_EIND_MODEL  = os.path.join(BASE_DIR, "hybrid_v3_EIND.keras")
PAD_CSV_LOG     = os.path.join(BASE_DIR, "training_log_hybrid_v3.csv")
PAD_HISTORY_NPY = os.path.join(BASE_DIR, "training_history_hybrid_v3.npy")
PAD_PLOT        = os.path.join(BASE_DIR, "hybrid_v3_training_plot.png")

print(f"  Checkpoints → {CKPT_DIR}")

def process_eeg_file(npz_filename, mode='cnn'):
    data           = np.load(npz_filename)
    eeg_data       = data['eeg']
    fs             = int(data['fs'])
    attended_wav   = str(data['stimulus_attended'])
    unattended_wav = str(data['stimulus_unattended'])

    if mode == 'cnn':
        target_sr, lowcut, highcut = 64, 1.0, 32.0
    elif mode == 'linear':
        target_sr, lowcut, highcut = 20, 1.0, 9.0
    else:
        raise ValueError("Kies 'linear' of 'cnn'")

    sos             = signal.butter(N=4, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')
    eeg_filtered    = signal.sosfiltfilt(sos, eeg_data, axis=0)
    g               = gcd(fs, target_sr)
    eeg_downsampled = signal.resample_poly(eeg_filtered, target_sr // g, fs // g, axis=0)

    return eeg_downsampled, attended_wav, unattended_wav

def batch_equalizer(eeg, env_1, env_2, labels):
    return (
        np.concatenate([eeg,   eeg  ], axis=0),
        np.concatenate([env_1, env_2], axis=0),
        np.concatenate([env_2, env_1], axis=0)
    ), np.concatenate([labels, (labels + 1) % 2], axis=0)

# DataGenerator MET overlap
class DataGenerator:
    """
    training=True  50% overlappende windows (hop = 320 samples)
                    
                     
    training=False niet-overlappend (eerlijke validatie/test)
    """
    def __init__(self, files, audio_dir, time_window=640, training=False):
        self.files       = list(files)
        self.audio_dir   = audio_dir
        self.time_window = time_window
        self.training    = training
        self.hop         = time_window // 2 if training else time_window

        print(f"   (training={training}, hop={self.hop})")
        alle_audio      = glob.glob(os.path.join(self.audio_dir, "**", "*.npy"), recursive=True)
        self.audio_dict = {os.path.basename(f).lower(): f for f in alle_audio}
        print(f"   {len(self.audio_dict)} audiobestanden gevonden")

    def __len__(self):
        return len(self.files)

    def _window_split(self, arr, expand_last=False):
        T = arr.shape[0]
        starts  = list(range(0, T - self.time_window + 1, self.hop))
        windows = np.stack([arr[s:s + self.time_window] for s in starts], axis=0)
        if expand_last:
            windows = np.expand_dims(windows, axis=-1)
        return windows

    def __getitem__(self, idx):
        eeg_processed, att_naam, unatt_naam = process_eeg_file(self.files[idx], mode='cnn')

        zoek_att   = f"{att_naam.replace('.wav','').replace('.npy','').strip().lower()}_cnn.npy"
        zoek_unatt = f"{unatt_naam.replace('.wav','').replace('.npy','').strip().lower()}_cnn.npy"

        att_pad   = self.audio_dict.get(zoek_att)
        unatt_pad = self.audio_dict.get(zoek_unatt)
        if not att_pad or not unatt_pad:
            raise FileNotFoundError(f"Audio '{zoek_att}' of '{zoek_unatt}' niet gevonden!")

        env1 = np.load(att_pad)
        env2 = np.load(unatt_pad)

        min_len = min(eeg_processed.shape[0], env1.shape[0], env2.shape[0])
        eeg_processed = eeg_processed[:min_len].astype(np.float32)
        env1 = env1[:min_len].astype(np.float32)
        env2 = env2[:min_len].astype(np.float32)

        eeg_w  = self._window_split(eeg_processed)
        env1_w = self._window_split(env1,  expand_last=True)
        env2_w = self._window_split(env2,  expand_last=True)

        n = eeg_w.shape[0]
        labels = np.ones((n, 1), dtype=np.float32)
        return batch_equalizer(eeg_w, env1_w, env2_w, labels)

    def __call__(self):
        if self.training:
            np.random.shuffle(self.files)
        for idx in range(len(self)):
            try:
                yield self.__getitem__(idx)
            except Exception as e:
                print(f"  Fout bij trial {idx}: {e}")
                continue


def maak_tf_dataset(files, audio_dir, time_window=640, training=False):
    gen = DataGenerator(files=files, audio_dir=audio_dir,
                        time_window=time_window, training=training)
    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            (tf.TensorSpec(shape=(None, time_window, 64), dtype=tf.float32),
             tf.TensorSpec(shape=(None, time_window,  1), dtype=tf.float32),
             tf.TensorSpec(shape=(None, time_window,  1), dtype=tf.float32)),
            tf.TensorSpec(shape=(None, 1), dtype=tf.float32)
        )
    )
    return ds.prefetch(tf.data.AUTOTUNE)


def bouw_hybrid_v3(
    time_window      = 640,
    n_layers         = 3,
    kernel_size      = 3,
    spatial_filters  = 8,   
    dilated_filters  = 16,  
    lstm_units       = 16,  
):
    eeg  = tf.keras.layers.Input(shape=[time_window, 64], name="EEG_Input")
    env1 = tf.keras.layers.Input(shape=[time_window,  1], name="Env1_Input")
    env2 = tf.keras.layers.Input(shape=[time_window,  1], name="Env2_Input")

    x = tf.keras.layers.BatchNormalization(name="EEG_BN_Input")(eeg)
    x = tf.keras.layers.Conv1D(spatial_filters, kernel_size=1,
                                padding='same', name="Spatial_Conv")(x)
    x = tf.keras.layers.BatchNormalization(name="EEG_BN_Spatial")(x)

    for l in range(n_layers):
        dil = kernel_size ** l
        x = tf.keras.layers.Conv1D(
            dilated_filters, kernel_size=kernel_size,
            dilation_rate=dil, padding='same',
            activation='relu', name=f"EEG_Dilated_L{l}_D{dil}"
        )(x)
        x = tf.keras.layers.BatchNormalization(name=f"EEG_BN_Dilated_L{l}")(x)

    eeg_out = x

    a1 = tf.keras.layers.BatchNormalization(name="Audio_BN_Input")(env1)
    a2 = tf.keras.layers.BatchNormalization(name="Audio_BN_Input_2")(env2)

    for l in range(n_layers):
        dil = kernel_size ** l
        shared_conv = tf.keras.layers.Conv1D(
            dilated_filters, kernel_size=kernel_size,
            dilation_rate=dil, padding='same',
            activation='relu', name=f"Audio_Dilated_L{l}_D{dil}"
        )
        shared_bn = tf.keras.layers.BatchNormalization(name=f"Audio_BN_Dilated_L{l}")
        a1 = shared_bn(shared_conv(a1))
        a2 = shared_bn(shared_conv(a2))

    shared_lstm = tf.keras.layers.LSTM(
        lstm_units, return_sequences=True,
        activation='tanh', name="Audio_LSTM_Shared"
    )
    audio_out1 = shared_lstm(a1)
    audio_out2 = shared_lstm(a2)

    cos1 = tf.keras.layers.Dot(axes=1, normalize=True, name="Cosine_Env1")([eeg_out, audio_out1])
    cos2 = tf.keras.layers.Dot(axes=1, normalize=True, name="Cosine_Env2")([eeg_out, audio_out2])

    flat = tf.keras.layers.Flatten()(tf.keras.layers.Concatenate()([cos1, cos2]))
    out  = tf.keras.layers.Dense(1, activation='sigmoid', name='output_name')(flat)

    model = tf.keras.Model(inputs=[eeg, env1, env2], outputs=[out])
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return model

TIME_WINDOW = 640

alle_eeg = glob.glob(PAD_TRAIN_EEG)
print(f"   Totaal EEG trials gevonden: {len(alle_eeg)}")

random.seed(SEED)
random.shuffle(alle_eeg)
split_idx   = int(len(alle_eeg) * 0.8)
train_files = alle_eeg[:split_idx]
val_files   = alle_eeg[split_idx:]
print(f"   Train: {len(train_files)} | Val: {len(val_files)}")

train_dataset = maak_tf_dataset(train_files, PAD_AUDIO, time_window=TIME_WINDOW, training=True)
val_dataset   = maak_tf_dataset(val_files,   PAD_AUDIO, time_window=TIME_WINDOW, training=False)

ckpt_files = glob.glob(os.path.join(CKPT_DIR, "*.keras"))

if not ckpt_files:
    raise ValueError(f"Geen checkpoints gevonden in {CKPT_DIR}")

laatste_checkpoint = sorted(ckpt_files)[-1]

print(f"   Automatisch hervatten vanaf: {laatste_checkpoint}")
model = tf.keras.models.load_model(laatste_checkpoint)

model.summary()
print(f"\n   Trainable parameters: {sum(tf.keras.backend.count_params(w) for w in model.trainable_weights):,}")


checkpoint_per_epoch = ModelCheckpoint(
    filepath=PAD_CHECKPOINT_EPOCH,
    monitor='val_accuracy',
    save_best_only=False,
    save_weights_only=False,
    verbose=1
)

checkpoint_best = ModelCheckpoint(
    filepath=PAD_BEST_MODEL,
    monitor='val_accuracy',
    save_best_only=True,
    save_weights_only=False,
    verbose=1
)

early_stop = EarlyStopping(
    monitor='val_accuracy',
    patience=15,
    restore_best_weights=True,
    verbose=1
)

# LearningRate verlagen bij plateau, helpt het model om zijn optimum fijn af te stellen
reduce_lr = ReduceLROnPlateau(
    monitor='val_loss',
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1
)

csv_logger = CSVLogger(PAD_CSV_LOG, append=True)

callbacks = [checkpoint_per_epoch, checkpoint_best, early_stop, reduce_lr, csv_logger]



start_epoch = 0

history = model.fit(
    train_dataset,
    validation_data=val_dataset,
    epochs=30,
    initial_epoch=start_epoch,
    callbacks=callbacks,
    verbose=1
)

model.save(PAD_EIND_MODEL)
np.save(PAD_HISTORY_NPY, history.history)
print(f"\n Eindmodel opgeslagen: {PAD_EIND_MODEL}")
print(f" Beste (single) model: {PAD_BEST_MODEL}")
print(f" Alle epochs in: {CKPT_DIR}")



import os, re, glob
import numpy as np
import tensorflow as tf

BASE_DIR     = "/content/drive/MyDrive/p&d/fase_2/"
CKPT_DIR     = os.path.join(BASE_DIR, "checkpoints_v3/")
PAD_TEST_EEG = os.path.join(BASE_DIR, "data/test_data/test_data/*/*.npz")
PAD_AUDIO    = os.path.join(BASE_DIR, "data/preprocessed/audio/cnn/")
TIME_WINDOW  = 640

def vind_beste_checkpoint(ckpt_dir):
    files = glob.glob(os.path.join(ckpt_dir, "*.keras"))
    if not files:
        raise ValueError(f"Geen checkpoints in {ckpt_dir}")
    parsed = []
    for f in files:
        m = re.search(r'valacc([\d\.]+)', os.path.basename(f))
        if m:
            parsed.append((float(m.group(1)), f))
    if not parsed:
        raise ValueError("Geen bestanden met 'valacc' in de naam gevonden")
    parsed.sort(reverse=True)
    return parsed[0]  # (val_acc, filepath)

best_val_acc, best_ckpt = vind_beste_checkpoint(CKPT_DIR)
print(f"   Beste checkpoint:")
print(f"   Bestand        : {os.path.basename(best_ckpt)}")
print(f"   Val-accuracy   : {best_val_acc*100:.2f}%\n")

# Laad dat model

model = tf.keras.models.load_model(best_ckpt)
print(f"   Trainable params: {sum(tf.keras.backend.count_params(w) for w in model.trainable_weights):,}\n")

# Test-dataset opbouwen
test_files = glob.glob(PAD_TEST_EEG)
print(f"  Test-trials gevonden: {len(test_files)}")

test_dataset = maak_tf_dataset(
    test_files,
    PAD_AUDIO,
    time_window=TIME_WINDOW,
    training=False  # geen overlap bij evaluatie
)

test_loss, test_acc = model.evaluate(test_dataset, verbose=1)


print(f"  Checkpoint         : {os.path.basename(best_ckpt)}")
print(f"  Validatie-accuracy : {best_val_acc*100:.2f}%")
print(f"  Test-accuracy      : {test_acc*100:.2f}%")
print(f"  Test-loss          : {test_loss:.4f}")

acc       = history.history['accuracy']
val_acc   = history.history['val_accuracy']
loss_hist = history.history['loss']
val_loss  = history.history['val_loss']
epochs_r  = range(1, len(acc) + 1)

best_epoch    = int(np.argmax(val_acc)) + 1
best_val_acc  = val_acc[best_epoch - 1]
best_val_loss = val_loss[best_epoch - 1]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle('Hybrid v3 (Minimal + Overlap + Ensemble) — Trainingsresultaten',
             fontsize=14, fontweight='bold')

ax1.plot(epochs_r, acc,     'b-', label='Train Accuracy',      linewidth=2)
ax1.plot(epochs_r, val_acc, 'g-', label='Validation Accuracy', linewidth=2)
ax1.axvline(x=best_epoch, color='red', linestyle='--',
            label=f'Best Epoch ({best_epoch})', alpha=0.8)
ax1.scatter(best_epoch, best_val_acc, color='red', s=30, zorder=5)
ax1.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax1.text(0.05, 0.95,
         f" Best Val Acc: {best_val_acc*100:.2f}%\n(Epoch {best_epoch})",
         transform=ax1.transAxes, fontsize=11, verticalalignment='top',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                   edgecolor='red', alpha=0.9))
ax1.set_title('A. Accuracy', fontsize=13, fontweight='bold')
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Accuracy')
ax1.grid(True, linestyle='--', alpha=0.6); ax1.legend(loc='lower right')

ax2.plot(epochs_r, loss_hist, 'b-', label='Train Loss',      linewidth=2)
ax2.plot(epochs_r, val_loss,  'g-', label='Validation Loss', linewidth=2)
ax2.axvline(x=best_epoch, color='red', linestyle='--',
            label=f'Best Epoch ({best_epoch})', alpha=0.8)
ax2.scatter(best_epoch, best_val_loss, color='red', s=30, zorder=5)
ax2.xaxis.set_major_locator(ticker.MaxNLocator(integer=True))
ax2.text(0.05, 0.05,
         f" Lowest Val Loss: {best_val_loss:.4f}\n(Epoch {best_epoch})",
         transform=ax2.transAxes, fontsize=11, verticalalignment='bottom',
         bbox=dict(boxstyle='round,pad=0.5', facecolor='white',
                   edgecolor='red', alpha=0.9))
ax2.set_title('B. Loss', fontsize=13, fontweight='bold')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Binary Cross-Entropy Loss')
ax2.grid(True, linestyle='--', alpha=0.6); ax2.legend(loc='upper right')

plt.tight_layout()
plt.savefig(PAD_PLOT, dpi=150)
plt.show()
print(f"   Grafiek opgeslagen: {PAD_PLOT}")

evaluatie

In [ ]:

import os, re, glob
import numpy as np
import tensorflow as tf
from scipy import signal
from math import gcd

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR     = "/content/drive/MyDrive/p&d/fase_2/"
CKPT_DIR     = os.path.join(BASE_DIR, "checkpoints_v3/")
PAD_TEST_EEG = os.path.join(BASE_DIR, "data/test_data/test_data/*/*.npz")
PAD_AUDIO    = os.path.join(BASE_DIR, "data/preprocessed/audio/cnn/")
TIME_WINDOW  = 640

def process_eeg_file(npz_filename, mode='cnn'):
    data = np.load(npz_filename)
    eeg_data = data['eeg']
    fs = int(data['fs'])
    attended_wav   = str(data['stimulus_attended'])
    unattended_wav = str(data['stimulus_unattended'])
    target_sr, lowcut, highcut = 64, 1.0, 32.0
    sos = signal.butter(N=4, Wn=[lowcut, highcut], btype='bandpass', fs=fs, output='sos')
    eeg_filtered = signal.sosfiltfilt(sos, eeg_data, axis=0)
    g = gcd(fs, target_sr)
    eeg_downsampled = signal.resample_poly(eeg_filtered, target_sr // g, fs // g, axis=0)
    return eeg_downsampled, attended_wav, unattended_wav

def batch_equalizer(eeg, env_1, env_2, labels):
    return (
        np.concatenate([eeg,   eeg  ], axis=0),
        np.concatenate([env_1, env_2], axis=0),
        np.concatenate([env_2, env_1], axis=0)
    ), np.concatenate([labels, (labels + 1) % 2], axis=0)

class EvalGenerator:
    def __init__(self, files, audio_dir, time_window=640):
        self.files = list(files)
        self.time_window = time_window
        self.hop = time_window  # geen overlap bij evaluatie
        alle_audio = glob.glob(os.path.join(audio_dir, "**", "*.npy"), recursive=True)
        self.audio_dict = {os.path.basename(f).lower(): f for f in alle_audio}
        print(f" {len(self.audio_dict)} audiobestanden gevonden")

    def __len__(self):
        return len(self.files)

    def _window_split(self, arr, expand_last=False):
        T = arr.shape[0]
        starts = list(range(0, T - self.time_window + 1, self.hop))
        windows = np.stack([arr[s:s + self.time_window] for s in starts], axis=0)
        if expand_last:
            windows = np.expand_dims(windows, axis=-1)
        return windows

    def __getitem__(self, idx):
        eeg_processed, att_naam, unatt_naam = process_eeg_file(self.files[idx])
        zoek_att   = f"{att_naam.replace('.wav','').replace('.npy','').strip().lower()}_cnn.npy"
        zoek_unatt = f"{unatt_naam.replace('.wav','').replace('.npy','').strip().lower()}_cnn.npy"
        att_pad   = self.audio_dict.get(zoek_att)
        unatt_pad = self.audio_dict.get(zoek_unatt)
        if not att_pad or not unatt_pad:
            raise FileNotFoundError(f"Audio '{zoek_att}' of '{zoek_unatt}' niet gevonden")
        env1 = np.load(att_pad);  env2 = np.load(unatt_pad)
        min_len = min(eeg_processed.shape[0], env1.shape[0], env2.shape[0])
        eeg_processed = eeg_processed[:min_len].astype(np.float32)
        env1 = env1[:min_len].astype(np.float32)
        env2 = env2[:min_len].astype(np.float32)
        eeg_w  = self._window_split(eeg_processed)
        env1_w = self._window_split(env1, expand_last=True)
        env2_w = self._window_split(env2, expand_last=True)
        n = eeg_w.shape[0]
        labels = np.ones((n, 1), dtype=np.float32)
        return batch_equalizer(eeg_w, env1_w, env2_w, labels)

    def __call__(self):
        for idx in range(len(self)):
            try:
                yield self.__getitem__(idx)
            except Exception as e:
                print(f"  Trial {idx} overgeslagen: {e}")
                continue

def maak_eval_dataset(files, audio_dir, time_window=640):
    gen = EvalGenerator(files, audio_dir, time_window)
    ds = tf.data.Dataset.from_generator(
        gen,
        output_signature=(
            (tf.TensorSpec(shape=(None, time_window, 64), dtype=tf.float32),
             tf.TensorSpec(shape=(None, time_window,  1), dtype=tf.float32),
             tf.TensorSpec(shape=(None, time_window,  1), dtype=tf.float32)),
            tf.TensorSpec(shape=(None, 1), dtype=tf.float32)
        )
    )
    return ds.prefetch(tf.data.AUTOTUNE)

files = glob.glob(os.path.join(CKPT_DIR, "*.keras"))
if not files:
    raise ValueError(f"Geen checkpoints gevonden in {CKPT_DIR}")

parsed = []
for f in files:
    m = re.search(r'valacc([\d\.]+)', os.path.basename(f))
    if m:
        parsed.append((float(m.group(1)), f))
parsed.sort(reverse=True)

best_val_acc, best_ckpt = parsed[0]
print(f"   Beste checkpoint op validatie:")
print(f"   Bestand      : {os.path.basename(best_ckpt)}")
print(f"   Val-accuracy : {best_val_acc*100:.2f}%\n")


model = tf.keras.models.load_model(best_ckpt)
n_params = sum(tf.keras.backend.count_params(w) for w in model.trainable_weights)
print(f"   Trainable params: {n_params:,}\n")

test_files = glob.glob(PAD_TEST_EEG)
print(f" Test-trials gevonden: {len(test_files)}")
test_dataset = maak_eval_dataset(test_files, PAD_AUDIO, time_window=TIME_WINDOW)


test_loss, test_acc = model.evaluate(test_dataset, verbose=1)


print(f"  Checkpoint         : {os.path.basename(best_ckpt)}")
print(f"  Validatie-accuracy : {best_val_acc*100:.2f}%")
print(f"  Test-accuracy      : {test_acc*100:.2f}%")
